In [1]:
# Bull/Bear Market Regime Analysis - reloading locked Test-period results
# No new data pulled, no recomputation of DTW/OCP/TOP/naive 
# this notebook only reslices and re-analyzes returns already computed and saved

import pandas as pd
import numpy as np
import pickle
import math
from statsmodels.stats.multitest import multipletests

RISK_FREE_RATE = 0.045
TEST_START = '2018-01-01'
TEST_END = '2025-12-31'

# Loading the exact, locked Test-period return series from the original notebook
with open('test_backtest_results_sp500_20y.pkl','rb') as f:
    test_method_portfolio_results = pickle.load(f)

with open('naive_buyhold_returns_sp500_20y.pkl', 'rb') as f:
    naive_data = pickle.load(f)
    benchmark_returns = naive_data['benchmark_returns']

naive_test_returns = benchmark_returns.loc[TEST_START:TEST_END]

print('Loaded strategies:', list(test_method_portfolio_results.keys()))
for method in test_method_portfolio_results:
    r = test_method_portfolio_results[method]['returns']
    print(f'  {method}: {len(r)} weeks, {r.index[0].date()} to {r.index[-1].date()}')

print(f'\nNaive Buy-Hold: {len(naive_test_returns)} weeks, '
      f'{naive_test_returns.index[0].date()} to {naive_test_returns.index[-1].date()}')

Loaded strategies: ['DTW', 'OCP', 'TOP']
  DTW: 418 weeks, 2018-01-03 to 2025-12-31
  OCP: 418 weeks, 2018-01-03 to 2025-12-31
  TOP: 418 weeks, 2018-01-03 to 2025-12-31

Naive Buy-Hold: 418 weeks, 2018-01-03 to 2025-12-31


In [14]:
# Defining bear-market window from independently published market history

BEAR_WINDOWS = [
    ('2018-10-01', '2018-12-31'),  # Q4 2018 correction
    ('2020-02-19', '2020-03-23'),  # COVID crash
    ('2022-01-03', '2022-10-12'),  # 2022 bear market
]

def label_regime(index):
    """
    Labels each date as 'bear' if it falls within any of the defined
    bear-market windows, otherwise 'bull'. Everything not explicitly
    flagged as bear defaults to bull. This is a bifurcation, not an
    attempt to also isolate a separate "normal" middle regime.
    """
    labels = pd.Series('bull', index=index)
    for start, end in BEAR_WINDOWS:
        labels[(index >= start) & (index <= end)] = 'bear'
    return labels

all_series = {
    'DTW': test_method_portfolio_results['DTW']['returns'],
    'OCP': test_method_portfolio_results['OCP']['returns'],
    'TOP': test_method_portfolio_results['TOP']['returns'],
    'Naive': naive_test_returns
}

regime_labels = label_regime(all_series['DTW'].index)
n_bear = (regime_labels == 'bear').sum()
n_bull = (regime_labels == 'bull').sum()
print(f'Regime split: {n_bear} bear-market weeks, {n_bull} bull-market weeks (of {len(regime_labels)} total)')

regime_returns = {'bear': {}, 'bull': {}}
for name, series in all_series.items():
    regime_returns['bear'][name] = series[regime_labels == 'bear']
    regime_returns['bull'][name] = series[regime_labels == 'bull']

print('\nBear-market week counts by strategy (should all match):')
for name in all_series:
    print(f"  {name}: {len(regime_returns['bear'][name])} bear weeks, {len(regime_returns['bull'][name])} bull weeks")

Regime split: 59 bear-market weeks, 359 bull-market weeks (of 418 total)

Bear-market week counts by strategy (should all match):
  DTW: 59 bear weeks, 359 bull weeks
  OCP: 59 bear weeks, 359 bull weeks
  TOP: 59 bear weeks, 359 bull weeks
  Naive: 59 bear weeks, 359 bull weeks


In [16]:
# Metrics functions (copy and pasted from previous notebook)
def compute_backtest_metrics(returns_series, rf_annual=RISK_FREE_RATE):
    """
    Computes standard backtest metrics from a weekly return series.
    """
    returns_series = returns_series.dropna()

    cumulative = returns_series.cumsum()
    total_return = cumulative.iloc[-1] if len(cumulative) > 0 else 0.0

    weekly_rf = rf_annual / 52
    excess_returns = returns_series - weekly_rf
    sharpe = (excess_returns.mean() / excess_returns.std()) * np.sqrt(52) if excess_returns.std() > 0 else 0.0

    running_max = cumulative.cummax()
    drawdown = cumulative - running_max
    max_drawdown = drawdown.min()

    n_trades = (returns_series != 0).sum()
    win_rate = (returns_series > 0).sum() / n_trades if n_trades > 0 else 0

    return {
        'total_return': total_return,
        'sharpe': sharpe,
        'max_drawdown': max_drawdown,
        'win_rate': win_rate,
        'n_active_weeks': n_trades
    }
    